# Create a Simple Reflex-Based Lunar Lander Agent

In this example, we will use Gymnasium, an environment to train agents via reinforcement learning (RL). We will not use RL here but just use the environment with a custom simple reflex-based agent.

## Install Gymnasium

The documentation for Gymnasium is available at https://gymnasium.farama.org/

Steps:
1. Create a new folder and open it with VS Code and install all needed Python Extensions in VS Code.
2. Create a new virtual environment (CTRL-Shift P Python Create Environment...)
3. I needed to install swig and the Python C++ headers on WSL2 via the terminal
    * `sudo apt install swig`
    * `sudo apt-get install python3-dev`
4. Install gymnasium with the needed extras

In [5]:
%pip install -q swig
%pip install -q "gymnasium[classic_control]" box2d pygame

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


## The Lunar Lander Environment

The documentation of the environment is available at: https://gymnasium.farama.org/environments/box2d/lunar_lander/

* Performance Measure: A reward of -100 or +100 points for crashing or landing safely respectively. We do not use
  intermediate rewards here.

* Environment: This environment is a classic rocket trajectory optimization problem. A ship needs to land safely. The space is **continuous** with
  x and y coordinates in the range [-2.5, 2.5]. The landing pad is at coordinate (0,0).

* Actuators:  According to Pontryagin’s
  maximum principle, it is optimal to fire the engine at full throttle or turn it off. This is the reason why this environment has discrete actions: engine on or off. There are four discrete actions available:

    - 0: do nothing
    - 1: fire left orientation engine
    - 2: fire main engine
    - 3: fire right orientation engine

* Sensors: Each observation is an 8-dimensional vector: the coordinates of the lander in x & y, its linear velocities in x & y, its angle, its angular velocity, and two booleans that represent whether each leg is in contact with the ground or not.

Gymnasim environments are implemented as classes with a `make` method to create the environment, a `reset` method, and a `step` method to execute an action.
To use it with an agent function that expects percetps and returns an action, we need write glue code that connects the environment with the agent function.

In [6]:
import gymnasium as gym

def run_episode(agent_function, max_steps=1000):
    """Run one episode in the LunarLander-v3 environment using the provided agent."""

    # Initialize the environment
    env = gym.make("LunarLander-v3", render_mode="human")

    # Reset the environment to generate the first observation (use seed=42 in reset to get reproducible results)
    observation, info = env.reset()

    # run one episode
    for _ in range(max_steps):
        # call the agent function to select an action
        action = agent_function(observation)

        print (f"Obs: {observation} -> Action: {action}")

        # step: execute an action in the environment
        observation, reward, terminated, truncated, info = env.step(action)

        env.render()

        if terminated:
            print(f"Final Reward: {reward}")
            break

    env.close()
    return reward

Note: `env.render()` shows the environment when the notebook is locally run (e.g., in VScode). On Colab, you cannot see the environment because the code is run on a headless server (i.e., a server without a display). There are some workarounds you can google.

## Example: A Random Agent

We ranomly return one of the actions. The environment accepts the integers 0-3.


In [7]:
import numpy as np

def random_agent_function(observation):
    """A random agent that selects actions uniformly at random. It ignores the observation."""
    return np.random.choice([0, 1, 2, 3], p=[0.25, 0.25, 0.25, 0.25])

run_episode(random_agent_function)

d:\Vscode\lunar_lander\.venv\Lib\site-packages\pygame\pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


Obs: [-0.00750837  1.4192805  -0.7605376   0.37154305  0.00870715  0.17227302
  0.          0.        ] -> Action: 0
Obs: [-0.01501732  1.4270632  -0.75950897  0.3458295   0.01721993  0.17027168
  0.          0.        ] -> Action: 3
Obs: [-0.02245626  1.4342438  -0.75071394  0.31904125  0.02395974  0.13480878
  0.          0.        ] -> Action: 1
Obs: [-0.02998629  1.4408227  -0.7621227   0.29222253  0.03298416  0.18050528
  0.          0.        ] -> Action: 3
Obs: [-0.03743906  1.446812   -0.7524241   0.266009    0.04005071  0.14134426
  0.          0.        ] -> Action: 1
Obs: [-0.04498482  1.4521965  -0.7640709   0.23902868  0.04945166  0.18803607
  0.          0.        ] -> Action: 0
Obs: [-0.05253096  1.4569819  -0.7640996   0.21235374  0.05884964  0.18797688
  0.          0.        ] -> Action: 1
Obs: [-0.06013765  1.4611737  -0.7716731   0.18583392  0.06975364  0.21809998
  0.          0.        ] -> Action: 3
Obs: [-0.06765614  1.4647866  -0.7605738   0.16015062  0.0784027

-100

## A Simple Reflex-Based Agent

To make the code easier to read, we use enumerations for actions (integers) and observations (index in the observation vector).

In [11]:
from enum import Enum

class Act(Enum):
    LEFT = 1
    RIGHT = 3
    MAIN = 2
    NO_OP = 0

class Obs(Enum):
    X = 0
    Y = 1
    VX = 2
    VY = 3
    ANGLE = 4
    ANGULAR_VELOCITY = 5
    LEFT_LEG_CONTACT = 6
    RIGHT_LEG_CONTACT = 7


In [12]:
def rocket_agent_function(observation):
    """A simple agent function."""

    # run the main thruster, if the lander is falling too fast
    if observation[Obs.VY.value] < -.3:
        return Act.MAIN.value

    return Act.NO_OP.value

run_episode(rocket_agent_function)

Obs: [-0.0046257   1.418057   -0.4685411   0.31718814  0.00536675  0.10613167
  0.          0.        ] -> Action: 0
Obs: [-0.00925179  1.4246161  -0.4679071   0.29148284  0.01061125  0.10489988
  0.          0.        ] -> Action: 0
Obs: [-0.01387796  1.4305763  -0.4679235   0.26485166  0.01585258  0.10483632
  0.          0.        ] -> Action: 0
Obs: [-0.01850433  1.4359367  -0.46793905  0.23817892  0.02109325  0.10482309
  0.          0.        ] -> Action: 0
Obs: [-0.02313099  1.4406976  -0.4679544   0.21150824  0.0263331   0.10480656
  0.          0.        ] -> Action: 0
Obs: [-0.02775774  1.4448587  -0.46796983  0.18483792  0.03157209  0.10478975
  0.          0.        ] -> Action: 0
Obs: [-0.03238468  1.4484202  -0.4679852   0.15816766  0.03681029  0.10477334
  0.          0.        ] -> Action: 0
Obs: [-0.03701172  1.451382   -0.46800056  0.13149725  0.04204764  0.10475668
  0.          0.        ] -> Action: 0
Obs: [-0.04163895  1.4537442  -0.46801582  0.10482673  0.0472841

-100

## Evaluating the Agent

Run the agent on 100 problems and report the average reward.

In [14]:
import numpy as np

def run_episode_test(agent_function):
    """Run one episode in the LunarLander-v3 environment using the provided agent."""

    # Initialise the environment
    env = gym.make("LunarLander-v3", render_mode=None)

    # Reset the environment to generate the first observation
    observation, info = env.reset()

    # run one episode (max. 1000 steps)
    for _ in range(1000):
        # call the agent to select an action
        action = agent_function(observation)

        # step (transition) through the environment with the action
        observation, reward, terminated, truncated, info = env.step(action)

        if terminated:
            break

    env.close()
    return reward

def run_episodes(agent_function, n=1000):
    """Run multiple episodes with the given agent and return the rewards for each episode."""
    return [run_episode_test(agent_function) for _ in range(n)]

rewards = run_episodes(rocket_agent_function)
print(rewards)

print(f"Average reward: {np.average(rewards)}")
print(f"Success rate: {np.sum(np.array(rewards) == 100)}/{len(rewards)}")

[-100, -100, -100, -100, 100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, 100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,

This is not great!


## Implement A Better Reflex-Based Agent

Build a better that uses its right and left thrusters to land the craft (more) safely. Test your agent function using 100 problems.

In [6]:
import gymnasium as gym
import numpy as np
from enum import Enum

def run_episode(agent_function, max_steps=1000, render=False):

    env = gym.make("LunarLander-v3", render_mode="rgb_array" if render else None)
    observation, info = env.reset()
    total_reward = 0
    terminated = False
    truncated = False
    
    for step in range(max_steps):
        action = agent_function(observation)
        observation, reward, terminated, truncated, info = env.step(action)
        total_reward += reward
        
        if terminated or truncated:
            break
            
    env.close()
    return total_reward

class Act(Enum):
    LEFT = 1
    RIGHT = 3
    MAIN = 2
    NO_OP = 0

class Obs(Enum):
    X = 0
    Y = 1
    VX = 2
    VY = 3
    ANGLE = 4
    ANGULAR_VELOCITY = 5
    LEFT_LEG_CONTACT = 6
    RIGHT_LEG_CONTACT = 7

In [8]:
def better_reflex_agent_function(observation):
    y = observation[Obs.Y.value]
    vy = observation[Obs.VY.value]
    vx = observation[Obs.VX.value]
    angle = observation[Obs.ANGLE.value]
    
    VY_SAFE_LIMIT = -0.4 
    
    if vy < VY_SAFE_LIMIT:
        return Act.MAIN.value
    
    if y < 0.2 and vy < -0.1:
        return Act.MAIN.value

    ANGLE_THRESHOLD = 0.05
    
    if angle > ANGLE_THRESHOLD:
        return Act.RIGHT.value
    elif angle < -ANGLE_THRESHOLD:
        return Act.LEFT.value
    
    VX_THRESHOLD = 0.2
    
    if vx > VX_THRESHOLD:
        return Act.LEFT.value
    elif vx < -VX_THRESHOLD:
        return Act.RIGHT.value
        
    return Act.NO_OP.value

In [11]:
N_PROBLEMS = 100 
rewards = []
    
for _ in range(N_PROBLEMS):
    reward = run_episode(better_reflex_agent_function, max_steps=1000) 
    rewards.append(reward)

print(f"Average reward: {np.average(rewards)}")
print(f"Success rate: {np.sum(np.array(rewards) >= 100)}/{len(rewards)}")

Average reward: 119.16867134017754
Success rate: 70/100
